# Pilot Assignment in Pure Python
## Five Exact Methods for Book Problem 2.6

This notebook presents five exact approaches for the pilot assignment problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve the base assignment model from book section `2.6`.

Each method reports:
- one optimal assignment
- the total skill score
- the execution time


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import permutations
from time import perf_counter


In [2]:
def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def score_assignment(assignment, scores):
    return sum(scores[pilot][craft] for pilot, craft in enumerate(assignment))


def assignment_to_solution(assignment, pilots, crafts, scores):
    mapping = {pilots[i]: crafts[assignment[i]] for i in range(len(pilots))}
    mapping["skill_score"] = score_assignment(assignment, scores)
    return mapping


def optimistic_bound(remaining_pilots, remaining_crafts, scores):
    return sum(max(scores[pilot][craft] for craft in remaining_crafts) for pilot in remaining_pilots)


## Problem: Pilot Assignment

Book section `2.6` evaluates four pilots on four aircraft types:
- dirigible
- airplane
- jet
- helicopter

The goal is to assign each pilot to exactly one aircraft, and each aircraft to exactly one pilot, maximizing the total exam score.


In [3]:
PILOTS = ["pilot_1", "pilot_2", "pilot_3", "pilot_4"]
CRAFTS = ["dirigible", "airplane", "jet", "helicopter"]
SCORES = [
    [6, 2, 8, 5],
    [9, 3, 5, 8],
    [4, 8, 3, 4],
    [6, 7, 6, 4],
]

EXPECTED = {
    "pilot_1": "jet",
    "pilot_2": "helicopter",
    "pilot_3": "airplane",
    "pilot_4": "dirigible",
    "skill_score": 30,
}


In [4]:
@timed
def solve_pilot_assignment_naive():
    best_assignment = None
    best_score = -1

    for assignment in permutations(range(len(CRAFTS))):
        current_score = score_assignment(assignment, SCORES)
        if current_score > best_score:
            best_score = current_score
            best_assignment = assignment

    return assignment_to_solution(best_assignment, PILOTS, CRAFTS, SCORES)


In [5]:
@timed
def solve_pilot_assignment_backtracking():
    best_assignment = None
    best_score = -1

    def backtrack(pilot_index, current_assignment, used_crafts, current_score):
        nonlocal best_assignment, best_score

        if pilot_index == len(PILOTS):
            if current_score > best_score:
                best_score = current_score
                best_assignment = tuple(current_assignment)
            return

        remaining_pilots = list(range(pilot_index + 1, len(PILOTS)))
        available_crafts = [craft for craft in range(len(CRAFTS)) if craft not in used_crafts]

        for craft in available_crafts:
            next_score = current_score + SCORES[pilot_index][craft]
            next_used = used_crafts | {craft}
            future_crafts = [c for c in range(len(CRAFTS)) if c not in next_used]
            bound = next_score + optimistic_bound(remaining_pilots, future_crafts, SCORES)
            if bound <= best_score:
                continue
            current_assignment.append(craft)
            backtrack(pilot_index + 1, current_assignment, next_used, next_score)
            current_assignment.pop()

    backtrack(0, [], set(), 0)
    return assignment_to_solution(best_assignment, PILOTS, CRAFTS, SCORES)


In [6]:
@timed
def solve_pilot_assignment_reduced_search():
    # If pilot 3 is not assigned to the airplane, the score cannot exceed 28.
    # If pilot 1 is not assigned to the jet after that, the score cannot exceed 29.
    forced = {2: 1, 0: 2}
    remaining_pilots = [1, 3]
    remaining_crafts = [0, 3]

    best_assignment = None
    best_score = -1
    for completion in permutations(remaining_crafts):
        assignment = [None] * 4
        for pilot, craft in forced.items():
            assignment[pilot] = craft
        for pilot, craft in zip(remaining_pilots, completion):
            assignment[pilot] = craft
        current_assignment = tuple(assignment)
        current_score = score_assignment(current_assignment, SCORES)
        if current_score > best_score:
            best_score = current_score
            best_assignment = current_assignment

    return assignment_to_solution(best_assignment, PILOTS, CRAFTS, SCORES)


In [7]:
@timed
def solve_pilot_assignment_dp():
    @lru_cache(maxsize=None)
    def dp(pilot_index, used_mask):
        if pilot_index == len(PILOTS):
            return 0, ()

        best_score = -1
        best_suffix = None
        for craft in range(len(CRAFTS)):
            if used_mask & (1 << craft):
                continue
            suffix_score, suffix_assignment = dp(pilot_index + 1, used_mask | (1 << craft))
            total_score = SCORES[pilot_index][craft] + suffix_score
            candidate = (craft,) + suffix_assignment
            if total_score > best_score:
                best_score = total_score
                best_suffix = candidate
        return best_score, best_suffix

    _, assignment = dp(0, 0)
    return assignment_to_solution(assignment, PILOTS, CRAFTS, SCORES)


In [8]:
@timed
def solve_pilot_assignment_branch_and_bound():
    best_assignment = None
    best_score = -1
    stack = [(0, tuple(), tuple(range(len(CRAFTS))), 0)]

    while stack:
        pilot_index, partial_assignment, remaining_crafts, current_score = stack.pop()

        if pilot_index == len(PILOTS):
            if current_score > best_score:
                best_score = current_score
                best_assignment = partial_assignment
            continue

        remaining_pilots = list(range(pilot_index + 1, len(PILOTS)))
        ordered_crafts = sorted(
            remaining_crafts,
            key=lambda craft: SCORES[pilot_index][craft],
            reverse=True,
        )

        for craft in ordered_crafts:
            next_score = current_score + SCORES[pilot_index][craft]
            next_remaining = tuple(c for c in remaining_crafts if c != craft)
            bound = next_score + optimistic_bound(remaining_pilots, next_remaining, SCORES)
            if bound <= best_score:
                continue
            stack.append((pilot_index + 1, partial_assignment + (craft,), next_remaining, next_score))

    return assignment_to_solution(best_assignment, PILOTS, CRAFTS, SCORES)


In [9]:
naive_result, naive_time = solve_pilot_assignment_naive()
backtracking_result, backtracking_time = solve_pilot_assignment_backtracking()
reduced_result, reduced_time = solve_pilot_assignment_reduced_search()
dp_result, dp_time = solve_pilot_assignment_dp()
bb_result, bb_time = solve_pilot_assignment_branch_and_bound()

print("=== PILOT ASSIGNMENT RESULTS ===")
print(f"Naive enumeration            -> {naive_result}, time = {naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {backtracking_result}, time = {backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {reduced_result}, time = {reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {dp_result}, time = {dp_time:.8f} seconds")
print(f"Branch and Bound             -> {bb_result}, time = {bb_time:.8f} seconds")

assert naive_result == EXPECTED
assert backtracking_result == naive_result
assert reduced_result == naive_result
assert dp_result == naive_result
assert bb_result == naive_result
print("All five exact methods agree with the textbook optimum.")


=== PILOT ASSIGNMENT RESULTS ===
Naive enumeration            -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}, time = 0.00001638 seconds
Backtracking with pruning    -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}, time = 0.00004358 seconds
Constraint-driven reduced search -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}, time = 0.00000538 seconds
Dynamic programming          -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}, time = 0.00002033 seconds
Branch and Bound             -> {'pilot_1': 'jet', 'pilot_2': 'helicopter', 'pilot_3': 'airplane', 'pilot_4': 'dirigible', 'skill_score': 30}, time = 0.00005367 seconds
All five exact methods agree with the textbook optimum.


## Variants in the Book

Section `2.6` does not include a separate student variation after the algebraic model, so this notebook closes with the base problem only.
